# 99. 데모 리소스 정리

02~06장에서 만든 리소스를 **역순으로** 삭제합니다. 각 단계는 실패해도 다음 단계로 진행합니다.

| 생성 챕터 | 리소스 |
|---|---|
| 06 | 온라인 평가 구성 (+ 툴킷이 만든 평가 실행 역할) |
| 05 | AgentCore Memory |
| 04 | Gateway(타깃·정책 엔진 연결), 정책·정책 엔진, Lambda, Gateway/Lambda IAM 역할 |
| 03 | API 키 자격 증명 공급자, Cognito 사용자 풀 |
| 02 | Runtime(+`PROD` 엔드포인트), ECR 리포지토리, 런타임 실행 역할 |

In [ ]:
import json, os, time
from pathlib import Path
import boto3
from botocore.exceptions import ClientError

from demo_helpers import config
from demo_helpers.aws_setup import get_region

REGION = get_region()
state = config.load_state()
control = boto3.client("bedrock-agentcore-control", region_name=REGION)
iam = boto3.client("iam")

def step(title, fn):
    try:
        fn()
        print(f"✅ {title}")
    except Exception as e:
        print(f"⚠️  {title}: {e}")

print(json.dumps(state, indent=2, ensure_ascii=False))

## 06장: 온라인 평가

In [ ]:
def delete_online_eval():
    if state.get("online_eval_config_id"):
        from bedrock_agentcore_starter_toolkit import Evaluation
        Evaluation(region=REGION).delete_online_config(state["online_eval_config_id"], delete_execution_role=True)

step("온라인 평가 구성 삭제", delete_online_eval)

## 05장: Memory

In [ ]:
def delete_memory():
    if state.get("memory_id"):
        control.delete_memory(memoryId=state["memory_id"])

step("Memory 삭제 요청", delete_memory)

## 04장: Gateway · Policy · Lambda

In [ ]:
def delete_gateway():
    gw_id = state.get("gateway_id")
    if not gw_id:
        return
    for t in control.list_gateway_targets(gatewayIdentifier=gw_id).get("items", []):
        control.delete_gateway_target(gatewayIdentifier=gw_id, targetId=t["targetId"])
    time.sleep(10)
    control.delete_gateway(gatewayIdentifier=gw_id)
    for _ in range(30):
        try:
            control.get_gateway(gatewayIdentifier=gw_id)
            time.sleep(5)
        except ClientError:
            break

def delete_policy_engine():
    pe_id = state.get("policy_engine_id")
    if not pe_id:
        return
    for p in control.list_policies(policyEngineId=pe_id).get("policies", []):
        control.delete_policy(policyEngineId=pe_id, policyId=p["policyId"])
    time.sleep(10)
    control.delete_policy_engine(policyEngineId=pe_id)

step("Gateway(타깃 포함) 삭제", delete_gateway)
step("정책 · 정책 엔진 삭제", delete_policy_engine)
step("Lambda 삭제", lambda: boto3.client("lambda", region_name=REGION).delete_function(FunctionName=config.LAMBDA_NAME))

## 03장: Identity · Cognito

In [ ]:
step("API 키 자격 증명 공급자 삭제",
     lambda: control.delete_api_key_credential_provider(name=config.PARTNER_API_PROVIDER))

def delete_cognito():
    if state.get("cognito_pool_id"):
        boto3.client("cognito-idp", region_name=REGION).delete_user_pool(UserPoolId=state["cognito_pool_id"])

step("Cognito 사용자 풀 삭제", delete_cognito)

## 02장: Runtime · ECR

In [ ]:
def delete_runtime():
    agent_id = state.get("agent_id")
    if not agent_id:
        return
    try:
        control.delete_agent_runtime_endpoint(agentRuntimeId=agent_id, endpointName="PROD")
        time.sleep(15)
    except ClientError as e:
        print("   (PROD 엔드포인트)", e.response["Error"]["Code"])
    control.delete_agent_runtime(agentRuntimeId=agent_id)

def delete_ecr():
    ecr = boto3.client("ecr", region_name=REGION)
    for repo in ecr.describe_repositories()["repositories"]:
        if repo["repositoryName"].startswith(f"bedrock-agentcore-{config.AGENT_NAME}"):
            ecr.delete_repository(repositoryName=repo["repositoryName"], force=True)
            print("   삭제:", repo["repositoryName"])

step("Runtime(+PROD 엔드포인트) 삭제", delete_runtime)
step("ECR 리포지토리 삭제", delete_ecr)

## IAM 역할

In [ ]:
def delete_role(role_name):
    for p in iam.list_role_policies(RoleName=role_name)["PolicyNames"]:
        iam.delete_role_policy(RoleName=role_name, PolicyName=p)
    for p in iam.list_attached_role_policies(RoleName=role_name)["AttachedPolicies"]:
        iam.detach_role_policy(RoleName=role_name, PolicyArn=p["PolicyArn"])
    iam.delete_role(RoleName=role_name)

for role in (f"{config.RUNTIME_ROLE_NAME_PREFIX}-{REGION}", config.GATEWAY_ROLE_NAME, config.LAMBDA_ROLE_NAME):
    step(f"IAM 역할 삭제: {role}", lambda r=role: delete_role(r))

## 로컬 파일

In [ ]:
for f in ["demo_state.json", ".bedrock_agentcore.yaml", "Dockerfile", ".dockerignore",
          "local_agent.log", "local_mcp.log", "eval_results_alice.json"]:
    if Path(f).exists():
        Path(f).unlink()
        print("삭제:", f)

### 수동 확인이 필요한 항목
스타터 툴킷이 배포 과정에서 자동 생성한 다음 리소스는 다른 에이전트와 공유될 수 있어 자동 삭제하지 않습니다. 필요하면 콘솔에서 삭제하세요.
- IAM 역할 `AmazonBedrockAgentCoreSDKCodeBuild-*` , CodeBuild 프로젝트 `bedrock-agentcore-customer_support_agent-builder`
- S3 버킷 `bedrock-agentcore-codebuild-sources-<계정>-<리전>`
- CloudWatch 로그 그룹 `/aws/bedrock-agentcore/runtimes/*`, `/aws/lambda/customer-support-demo-tools`
- (선택) CloudWatch Transaction Search 설정 — 다른 워크로드가 사용하지 않는다면 X-Ray 콘솔에서 비활성화